# SWAN Skull Stripping with nnUNet
Assume you already have the SWAN NIfTIs and corresponding masks ready to train a nnUNet

## Set Export Path
## For the CLI commands below, please run in terminal with nnunet venv setup
Change the paths of nnUNet_preprocessed and nnUNet_results when you run and save

In [ ]:
conda create -n nnunet python=3.10 -y
conda activate nnunet

#git clone from nnUNet: https://github.com/MIC-DKFZ/nnUNet.git
cd /path/to/your/clonednnUNet
rm -rf wandb
pip install -e .
pip install numpy==1.26.4
pip install torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
export nnUNet_raw='/media/volume1/Luke/SWAN_SkullStripping_Data/nnunet'
export nnUNet_preprocessed='/media/volume1/Luke/SWAN_SkullStripping_nnUNet/exp1/nnUNet_preprocessed'
export nnUNet_results='/media/volume1/Luke/SWAN_SkullStripping_nnUNet/exp1/nnUNet_results'

In [ ]:
echo $nnUNet_raw
echo $nnUNet_preprocessed
echo $nnUNet_results

## Label Rename

In [ ]:
import os, shutil, gzip, re
from pathlib import Path
import nibabel as nib  # pip install nibabel
from tqdm import tqdm

# --- configure your folders ---
src_labels = "/media/volume1/Luke/SWAN_SkullStripping_Data/luke_raw"
dst_labels = "/media/volume1/Luke/SWAN_SkullStripping_Data/nnunet/Dataset702_SWANSkullStrip1121/labelsTr"


os.makedirs(dst_labels, exist_ok=True)

label_pat = re.compile(r"^mask(?P<pid>[^_]+)_(?P<sid>[^_]+)_SWAN\.(nii|nii\.gz)$", re.IGNORECASE)

def save_as_nii_gz(src_path: str, dst_path: str):
    """Load with nibabel and re-save to guarantee a valid .nii.gz readable by Slicer."""
    img = nib.load(src_path)
    nib.save(img, dst_path)

# -------- labels --------
for f in tqdm(os.listdir(src_labels)):
    m = label_pat.match(f)
    if not m: 
        continue
    pid, sid = m.group("pid"), m.group("sid")
    new_name = f"{pid}_{sid}_SWAN.nii.gz"   # NOTE: no _0000
    src = os.path.join(src_labels, f)
    dst = os.path.join(dst_labels, new_name)
    if f.lower().endswith(".nii.gz"):
        shutil.copy2(src, dst)
    else:
        save_as_nii_gz(src, dst)

print("Done. Check counts match: imagesTr == labelsTr.")

## Create a {Numbered} Dataset folder under your nnUNet_raw
Dataset<3-digit-ID> _ < Descriptive_Name > <br>
Then move your imagesTr, labelsTr within

## Set dataset.json
Reference for dataset.json: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/dataset_format.md#datasetjson

In [ ]:
import json
import os

# === Adjust numbers as needed ===
num_training_cases = len(os.listdir("/media/volume1/Luke/SWAN_SkullStripping_Data/nnunet/Dataset702_SWANSkullStrip1121/imagesTr"))

dataset_json = {
    "channel_names": {
        "0": "CT"
    },
    "labels": {
        "background": 0,
        "PE": 1
    },
    "numTraining": num_training_cases,
    "file_ending": ".nii.gz"
}

json_path = "/media/volume1/Luke/SWAN_SkullStripping_Data/nnunet/Dataset702_SWANSkullStrip1121/dataset.json"
os.makedirs(os.path.dirname(json_path), exist_ok=True)

with open(json_path, "w") as f:
    json.dump(dataset_json, f, indent=4)

print(f"✅ dataset.json created successfully at {json_path}")

## Extract Fingerprint
Change 505 to a number you choose

In [ ]:
nnUNetv2_plan_and_preprocess -d 702 --verify_dataset_integrity

## Start Training

In [ ]:
nnUNetv2_train 702 3d_fullres 0


## Start Inference

In [ ]:
nnUNetv2_predict \
  -d 702 \
  -c 3d_fullres \
  -f 0 \
  -i /media/volume1/Luke/Microbleed_Synthetic_Data/Tryout_1204/HealthyBrain/nifti \
  -o /media/volume1/Luke/Microbleed_Synthetic_Data/Tryout_1204/HealthyBrain/brainmask \
  --save_probabilities \
  -chk checkpoint_best.pth

## Clean up Brain Mask

In [ ]:
from pathlib import Path
import nibabel as nib
import numpy as np
from scipy import ndimage


def keep_largest_connected_component(mask, connectivity=26):
    if connectivity == 6:
        structure = ndimage.generate_binary_structure(3, 1)
    elif connectivity == 18:
        structure = ndimage.generate_binary_structure(3, 2)
    elif connectivity == 26:
        structure = ndimage.generate_binary_structure(3, 3)
    else:
        raise ValueError("connectivity must be 6, 18, or 26")

    labeled, num = ndimage.label(mask, structure=structure)

    if num == 0:
        return mask

    sizes = ndimage.sum(mask, labeled, index=np.arange(1, num + 1))
    largest_label = int(np.argmax(sizes)) + 1

    return labeled == largest_label


def postprocess_brainmask_nifti(
    input_path,
    output_path,
    threshold=0,
    connectivity=26,
    fill_holes_first=True,
):
    nii = nib.load(str(input_path))
    data = nii.get_fdata()

    mask = data > threshold

    if fill_holes_first:
        mask = ndimage.binary_fill_holes(mask)

    mask = keep_largest_connected_component(
        mask,
        connectivity=connectivity,
    )

    if not fill_holes_first:
        mask = ndimage.binary_fill_holes(mask)

    output_data = mask.astype(np.uint8)

    out_nii = nib.Nifti1Image(
        output_data,
        affine=nii.affine,
        header=nii.header,
    )

    out_nii.set_data_dtype(np.uint8)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    nib.save(out_nii, str(output_path))


def postprocess_brainmask_folder(
    input_dir,
    output_dir,
    threshold=0,
    connectivity=26,
    fill_holes_first=True,
):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    nifti_paths = sorted(
        list(input_dir.glob("*.nii")) +
        list(input_dir.glob("*.nii.gz"))
    )

    if len(nifti_paths) == 0:
        raise FileNotFoundError(f"No NIfTI files found in {input_dir}")

    for i, input_path in enumerate(nifti_paths, 1):
        output_path = output_dir / input_path.name

        print(f"[{i}/{len(nifti_paths)}] {input_path.name}")

        postprocess_brainmask_nifti(
            input_path=input_path,
            output_path=output_path,
            threshold=threshold,
            connectivity=connectivity,
            fill_holes_first=fill_holes_first,
        )

    print(f"\nDone. Postprocessed masks saved to:")
    print(output_dir)


if __name__ == "__main__":
    input_dir = "/media/volume1/Luke/Microbleed_Data/DataSource/HealthyBrain/batches/batch060326/brainmask"
    output_dir = "/media/volume1/Luke/Microbleed_Data/DataSource/HealthyBrain/batches/batch060326/cleaned_brainmask"

    postprocess_brainmask_folder(
        input_dir=input_dir,
        output_dir=output_dir,
        threshold=0,
        connectivity=26,
        fill_holes_first=True,
    )

## Apply Brain Masks and save Brain-Only Nifti

In [ ]:
from pathlib import Path
import nibabel as nib
import numpy as np


def apply_brain_mask(
    nifti_dir: Path,
    brainmask_dir: Path,
    out_dir: Path,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    nifti_paths = sorted(nifti_dir.glob("*_0000.nii.gz"))

    for nifti_path in nifti_paths:
        case_name = nifti_path.name.replace("_0000.nii.gz", "")
        mask_path = brainmask_dir / f"{case_name}.nii.gz"

        if not mask_path.exists():
            print(f"[WARN] Missing mask for {case_name}, skipping.")
            continue

        # --- load image ---
        img = nib.load(str(nifti_path))
        img_data = img.get_fdata()  # float64

        # --- load mask ---
        mask_img = nib.load(str(mask_path))
        mask_data = mask_img.get_fdata()

        # --- sanity check ---
        if img_data.shape != mask_data.shape:
            raise ValueError(f"Shape mismatch for {case_name}")

        # --- apply mask ---
        masked_data = img_data * (mask_data > 0)

        # --- save ---
        out_path = out_dir / nifti_path.name
        out_img = nib.Nifti1Image(
            masked_data.astype(img_data.dtype),
            affine=img.affine,
            header=img.header,
        )
        out_img.set_data_dtype(img_data.dtype)
        nib.save(out_img, str(out_path))

        print(f"[OK] Saved brain-masked NIfTI: {out_path.name}")


In [ ]:
apply_brain_mask(
    nifti_dir=Path("/media/volume1/Luke/Microbleed_Data/DataSource/TrueMCB/canonical_3D/images"),
    brainmask_dir=Path("/media/volume1/Luke/Microbleed_Data/DataSource/TrueMCB/canonical_3D/cleaned_brainmask"),
    out_dir=Path("/media/volume1/Luke/Microbleed_Data/DataSource/TrueMCB/canonical_3D/brainmasked_images"),
)

## Normalize Nifti Volumes: Apply volume-wise normalization to 255

In [31]:
from pathlib import Path
import numpy as np
import nibabel as nib


def normalize_nifti_dir_volumewise_to_u8(
    in_dir: Path,
    out_dir: Path,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    nifti_paths = sorted(in_dir.glob("*.nii.gz"))
    if not nifti_paths:
        raise RuntimeError(f"No NIfTI files found in {in_dir}")

    for nifti_path in nifti_paths:
        print(f"[INFO] Normalizing {nifti_path.name}")

        img = nib.load(str(nifti_path))
        vol = np.asarray(img.dataobj, dtype=np.float32)

        brain = vol[vol > 0]

        if brain.size == 0:
            norm_vol = np.zeros(vol.shape, dtype=np.uint8)
        else:
            p2, p98 = np.percentile(brain, [2, 98])

            vol = np.clip(vol, p2, p98)
            vol = (vol - p2) / (p98 - p2 + 1e-6)
            vol = vol * 255.0

            norm_vol = np.clip(np.rint(vol), 0, 255).astype(np.uint8)
            norm_vol[vol <= 0] = 0

        out_path = out_dir / nifti_path.name

        header = img.header.copy()
        header.set_data_dtype(np.uint8)

        out_img = nib.Nifti1Image(
            norm_vol,
            affine=img.affine,
            header=header,
        )

        nib.save(out_img, str(out_path))
        print(f"[OK] Saved {out_path.name}")

In [34]:
normalize_nifti_dir_volumewise_to_u8(
    in_dir=Path("/media/volume1/Luke/Microbleed_Data/DataSource/TrueMCB/canonical_3D/brainmasked_images"),
    out_dir=Path("/media/volume1/Luke/Microbleed_Data/DataSource/TrueMCB/canonical_3D/normalized_brainmasked_images"),
)

[INFO] Normalizing 101850_R3654600_SWAN_0000.nii.gz
[OK] Saved 101850_R3654600_SWAN_0000.nii.gz
[INFO] Normalizing 10186200_R3539548_SWAN_0000.nii.gz
[OK] Saved 10186200_R3539548_SWAN_0000.nii.gz
[INFO] Normalizing 10414774_R3634534_SWAN_0000.nii.gz
[OK] Saved 10414774_R3634534_SWAN_0000.nii.gz
[INFO] Normalizing 10425126_R3623203_SWAN_0000.nii.gz
[OK] Saved 10425126_R3623203_SWAN_0000.nii.gz
[INFO] Normalizing 10715154_R3649335_SWAN_0000.nii.gz
[OK] Saved 10715154_R3649335_SWAN_0000.nii.gz
[INFO] Normalizing 10849060_R3569815_SWAN_0000.nii.gz
[OK] Saved 10849060_R3569815_SWAN_0000.nii.gz
[INFO] Normalizing 10933112_R3596179_SWAN_0000.nii.gz
[OK] Saved 10933112_R3596179_SWAN_0000.nii.gz
[INFO] Normalizing 11001769_R3513714_SWAN_0000.nii.gz
[OK] Saved 11001769_R3513714_SWAN_0000.nii.gz
[INFO] Normalizing 11108876_R3680396_SWAN_0000.nii.gz
[OK] Saved 11108876_R3680396_SWAN_0000.nii.gz
[INFO] Normalizing 11135106_R3579962_SWAN_0000.nii.gz
[OK] Saved 11135106_R3579962_SWAN_0000.nii.gz
[INF

## Save Normalized, Brain-Masked 2D png

In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib
from PIL import Image


def nifti_folder_to_png_slices(
    src_dir,
    dst_dir,
    slice_axis=2,
    pad=4,
):
    """
    Convert all 3D NIfTI files in src_dir into per-slice PNGs.

    Input:
        src_dir/
            caseA.nii.gz
            caseB.nii.gz

    Output:
        dst_dir/
            caseA/
                caseA_0000.png
                caseA_0001.png
                ...
            caseB/
                caseB_0000.png
                ...

    Assumes NIfTI intensities are already normalized to [0, 255].
    """

    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    nifti_files = sorted(
        list(src_dir.glob("*.nii")) + list(src_dir.glob("*.nii.gz"))
    )

    if not nifti_files:
        raise FileNotFoundError(f"No NIfTI files found in {src_dir}")

    for nifti_path in nifti_files:

        case_name = nifti_path.stem
        if case_name.endswith(".nii"):   # handle .nii.gz
            case_name = case_name[:-4]

        out_case_dir = dst_dir / case_name
        out_case_dir.mkdir(parents=True, exist_ok=True)

        img = nib.load(str(nifti_path))
        data = img.get_fdata()

        if data.ndim != 3:
            raise ValueError(
                f"{nifti_path.name} is not 3D (shape={data.shape})"
            )

        # Move slice axis to the end → (H, W, S)
        data = np.moveaxis(data, slice_axis, -1)

        # Ensure valid PNG range
        data = np.clip(data, 0, 255).astype(np.uint8)

        for i in range(data.shape[-1]):

            slice_2d = data[..., i].T   # explicit transpose

            idx = str(i).zfill(pad) if pad is not None else str(i)

            out_path = out_case_dir / f"{case_name}_{idx}.png"

            Image.fromarray(slice_2d).save(out_path)

In [ ]:
nifti_folder_to_png_slices(
    src_dir="/media/volume1/Luke/Microbleed_Data/DataSource/HealthyBrain/canonical_3D/normalized_brainmasked_images",
    dst_dir="/media/volume1/Luke/Microbleed_Data/DataSource/HealthyBrain/canonical_2D/normalized_brainmasked_images",
    slice_axis=2
)